In [2]:
import sqlite3
import datetime
from datetime import timedelta, datetime
import pandas as pd
import datetime as dt

def read_database(db_path, table_name='itin'):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute(f'SELECT * FROM {table_name}')
    rows = cursor.fetchall()
    
    data = pd.DataFrame(rows, columns=[col[0] for col in cursor.description])

    conn.close()
    return data

def filter_itin(data, flight_number, date_till = None):
    if flight_number:
        data = data[data['itinerary_id'].str.contains(f'{flight_number}-', na=False)]
    if date_till:
        print(date_till)
        data['departure_date'] = pd.to_datetime(data['departure_date'], errors='coerce')
        data = data[data['departure_date'] <= date_till]
    
    return data



In [3]:
itin_df = read_database('flights.db')
legs_df = read_database('flights.db', table_name='legs')
prices_df = read_database('flights.db', table_name='prices')


In [4]:
# Filter itineraries for a specific flight number and date until today
filtered_itin_df = filter_itin(itin_df, 'FR103', dt.datetime.today())

# Merge the filtered itineraries with legs and prices dataframes
merged_df = pd.merge(filtered_itin_df, legs_df, on='itinerary_id', how='left')
merged_df2 = pd.merge(merged_df, prices_df, on='itinerary_id', how='left')


2025-07-08 22:02:17.877145


C:\Users\pinyc\AppData\Local\Temp\ipykernel_11144\725481060.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['departure_date'] = pd.to_datetime(data['departure_date'], errors='coerce')


In [5]:
# Feature engineering
proccessed_df = merged_df2.copy().loc[:, ['departure_datetime', 'arrival_datetime', 'days_to_departure','travel_time', 'aircraft','seat_pitch_type', 'price_datetime', 'day_of_week', 'price']]

# 1. Convert departure_datetime and arrival_datetime to datetime objects
proccessed_df['departure_datetime'] = pd.to_datetime(proccessed_df['departure_datetime'], format='mixed')
proccessed_df['arrival_datetime'] = pd.to_datetime(proccessed_df['arrival_datetime'], format='mixed')
proccessed_df['price_datetime'] = pd.to_datetime(proccessed_df['price_datetime'], format='mixed')

# 2. Convert aircraft and sseat_pitch_type to categorical variables
proccessed_df['aircraft'] = proccessed_df['aircraft'].astype('category')
proccessed_df['seat_pitch_type'] = proccessed_df['seat_pitch_type'].astype('category')

#3. Extract features from departure_datetime
proccessed_df['days_to_departure'] = (
   proccessed_df['departure_datetime'].dt.date - proccessed_df['price_datetime'].dt.date
).apply(lambda x: x.days)


proccessed_df['is_weekend'] = proccessed_df['departure_datetime'].dt.dayofweek >= 4  # 5=Saturday, 6=Sunday

# 3.2. Extract date and time features
#proccessed_df['dep_day'] = proccessed_df['departure_datetime'].dt.day
proccessed_df['dep_month'] = proccessed_df['departure_datetime'].dt.month
proccessed_df['departure_date'] = proccessed_df['departure_datetime'].dt.date
proccessed_df['dep_day_of_week'] = proccessed_df['departure_datetime'].dt.day_name()

bins = [0, 6, 12, 17, 24]
labels = ['Night', 'Morning', 'Afternoon', 'Evening']

proccessed_df['dep_hour'] = pd.cut(proccessed_df['departure_datetime'].dt.hour, bins=bins, labels=labels, right=False, include_lowest=True)
proccessed_df['price_hour'] = pd.cut(proccessed_df['price_datetime'].dt.hour, bins=bins, labels=labels, right=False, include_lowest=True)

proccessed_df1 = proccessed_df.loc[:, ['price','days_to_departure', 'departure_date', 'dep_month', 'dep_day_of_week', 'is_weekend', 'dep_hour', 'price_hour', 'aircraft', 'seat_pitch_type']]

In [6]:
proccessed_df1.head()

,price,days_to_departure,departure_date,dep_month,dep_day_of_week,is_weekend,dep_hour,price_hour,aircraft,seat_pitch_type
0,249.97,1,2025-05-04,5,Sunday,True,Afternoon,Afternoon,Boeing 737MAX 8 Passenger,below_average
1,249.97,1,2025-05-04,5,Sunday,True,Afternoon,Afternoon,Boeing 737MAX 8 Passenger,below_average
2,249.97,1,2025-05-04,5,Sunday,True,Afternoon,Evening,Boeing 737MAX 8 Passenger,below_average
3,97.99,1,2025-05-09,5,Friday,True,Morning,Afternoon,Boeing 737,average
4,121.99,1,2025-05-09,5,Friday,True,Morning,Evening,Boeing 737,average


In [7]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from datetime import datetime
from sklearn.model_selection import train_test_split

In [8]:
proccessed_df1.to_csv('processed_flights.csv', index=False)
import sklearn
print(sklearn.__version__)

1.7.0


In [9]:
data = proccessed_df1.copy()

# Convert departure_date to days since reference (e.g., 2025-01-01)
reference_date = datetime(2025, 1, 1)
data['departure_date_numeric'] = (pd.to_datetime(data['departure_date']) - reference_date).dt.days

# Encode categorical variables
categorical_cols = ['dep_day_of_week', 'dep_hour', 'price_hour', 'aircraft', 'seat_pitch_type']
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_cols = encoder.fit_transform(data[categorical_cols])
encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(categorical_cols))

# Combine features
X = pd.concat([data[['days_to_departure', 'departure_date_numeric', 'dep_month', 'is_weekend']], encoded_df], axis=1)
y = data['price']

# Split the dataset into training and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Verify the split
print(f"Training set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")

# Optional: Train and evaluate the model
model = xgb.XGBRegressor (objective='reg:squarederror', max_depth=10, n_estimators=100)
model.fit(X_train, y_train)

# Evaluate on test set (e.g., mean squared error)
from sklearn.metrics import mean_squared_error
y_pred = model.predict(X_test)
print(X_test)
# Compare predicted vs actual values
comparison_df = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred}, index=y_test.index)
print(comparison_df.head(10))
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on test set: {mse}")
print(f"Model accuracy: {model.score(X_test, y_test)}")

Training set size: 164 rows
Test set size: 42 rows
     days_to_departure  departure_date_numeric  dep_month  is_weekend  \
15                  15                     137          5        True   
9                    8                     135          5        True   
201                  5                     186          7        True   
82                  14                     163          6        True   
68                  11                     158          6        True   
97                  14                     165          6        True   
180                 11                     184          7        True   
163                  8                     179          6        True   
148                  4                     177          6        True   
183                  4                     184          7        True   
101                  5                     165          6        True   
165                  5                     179          6        True   


In [10]:
def price_will_go_down(df):
    df = df.sort_values(['departure_date', 'days_to_departure'], ascending=[True, False])
    df['label'] = 0

    for key, group in df.groupby('departure_date'):
        prices = group['price'].values

        # Add 10% trade-off: only count as flag=1 if a future price is at least 10% lower
        flag = [int(any(prices[j] < prices[i] * 0.9 for j in range(i+1, len(prices)))) for i in range(len(prices))]
        df.loc[group.index, 'label'] = flag

    return df


data = price_will_go_down(proccessed_df1)

In [11]:
# Convert departure_date to days since reference (e.g., 2025-01-01)
reference_date = datetime(2025, 1, 1)
data['departure_date_numeric'] = (pd.to_datetime(data['departure_date']) - reference_date).dt.days

# Encode categorical variables
categorical_cols = ['dep_day_of_week', 'dep_hour', 'price_hour', 'aircraft', 'seat_pitch_type']
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_cols = encoder.fit_transform(data[categorical_cols])
encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(categorical_cols))

# Combine features
X = pd.concat([data[['days_to_departure', 'price','departure_date_numeric', 'dep_month', 'is_weekend']], encoded_df], axis=1)
y = data['label']

# Split the dataset into training and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the split
print(f"Training set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")

# Optional: Train and evaluate the model
model = xgb.XGBClassifier(objective='binary:logistic', max_depth = 5, n_estimators=100, learning_rate=0.2, random_state=42)
model.fit(X_train, y_train)

# Evaluate on test set (e.g., mean squared error)
from sklearn.metrics import mean_squared_error
y_pred = model.predict(X_test)
print(y_pred)
# Compare predicted vs actual values
comparison_df = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred}, index=y_test.index)
#print(comparison_df.head(10))
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on test set: {mse}")
print(f"Model accuracy: {model.score(X_test, y_test)}")

print(X_test.columns)

Training set size: 164 rows
Test set size: 42 rows
[0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 0 1 1 0 0 0 1 1 0 0 1 0 1 0 0 0 0 1 0 1 1
 0 0 0 1 1]
Mean Squared Error on test set: 0.11904761904761904
Model accuracy: 0.8809523809523809
Index(['days_to_departure', 'price', 'departure_date_numeric', 'dep_month',
       'is_weekend', 'dep_day_of_week_Sunday', 'dep_hour_Morning',
       'price_hour_Evening', 'price_hour_Morning', 'price_hour_Night',
       'aircraft_Boeing 737MAX 8 Passenger', 'seat_pitch_type_below_average'],
      dtype='object')


In [12]:
flight_params = {
    'departure_date': datetime(2025, 7, 6),
    'departure_date_numeric': (datetime(2025, 7, 6) - datetime(2025, 1, 1)).days,
    'dep_month': 7,
    'is_weekend': 1,
    'dep_day_of_week_Sunday': 1,
    'dep_hour_Morning': 0,
    'price_hour_Evening': 1,
    'price_hour_Morning': 0,
    'price_hour_Night': 0,
    'aircraft_Boeing 737MAX 8 Passenger': 1,
    'seat_pitch_type_below_average': 0
}

test_flight = pd.read_csv('test_flight.csv')

test_rows = []

for row in test_flight.itertuples(index=False):
    ft = flight_params.copy()
    ft['days_to_departure'] = row.Date + 2
    ft['price'] = row.Price
    test_rows.append(ft)

# Create DataFrame with the same columns and order as X_train
test_df = price_will_go_down(pd.DataFrame(test_rows))

test_df.tail(10)

,departure_date,departure_date_numeric,dep_month,is_weekend,dep_day_of_week_Sunday,dep_hour_Morning,price_hour_Evening,price_hour_Morning,price_hour_Night,aircraft_Boeing 737MAX 8 Passenger,seat_pitch_type_below_average,days_to_departure,price,label
51,2025-07-06,186,7,1,1,0,1,0,0,1,0,11,73,0
52,2025-07-06,186,7,1,1,0,1,0,0,1,0,10,75,0
53,2025-07-06,186,7,1,1,0,1,0,0,1,0,9,75,0
54,2025-07-06,186,7,1,1,0,1,0,0,1,0,8,85,0
55,2025-07-06,186,7,1,1,0,1,0,0,1,0,7,86,0
56,2025-07-06,186,7,1,1,0,1,0,0,1,0,6,86,0
57,2025-07-06,186,7,1,1,0,1,0,0,1,0,5,110,0
58,2025-07-06,186,7,1,1,0,1,0,0,1,0,4,110,0
59,2025-07-06,186,7,1,1,0,1,0,0,1,0,3,153,0
60,2025-07-06,186,7,1,1,0,1,0,0,1,0,2,235,0


In [13]:
X = test_df[['days_to_departure', 'price', 'departure_date_numeric', 'dep_month',
       'is_weekend', 'dep_day_of_week_Sunday', 'dep_hour_Morning',
       'price_hour_Evening', 'price_hour_Morning', 'price_hour_Night',
       'aircraft_Boeing 737MAX 8 Passenger', 'seat_pitch_type_below_average']]


y = test_df['label']
print(y.head(10))
test_predict = model.predict(X)
print(pd.DataFrame({'Actual': y.values, 'Predicted': test_predict}, index=y.index).tail(20))
print(f"Model accuracy: {model.score(X, y)}")

0    0
1    0
2    0
3    0
4    1
5    1
6    1
7    1
8    1
9    1
Name: label, dtype: int64
    Actual  Predicted
41       1          0
42       1          1
43       1          0
44       1          1
45       1          1
46       1          0
47       1          0
48       1          0
49       1          0
50       0          0
51       0          0
52       0          0
53       0          0
54       0          0
55       0          0
56       0          0
57       0          0
58       0          0
59       0          0
60       0          0
Model accuracy: 0.8688524590163934
